# FIGURAS DE MÉRITO

In [2]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np
import cv2

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

def resize_for_hausdorff(pred, gt, max_size=512):
    if max(pred.shape) > max_size:
        scale = max_size / max(pred.shape)
        new_h = int(pred.shape[0] * scale)
        new_w = int(pred.shape[1] * scale)
        pred = cv2.resize(pred.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
        gt   = cv2.resize(gt.astype(np.uint8),   (new_w, new_h), interpolation=cv2.INTER_NEAREST).astype(bool)
    return pred, gt

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [3]:
import time
import torch

def measure_inference(model, img_path, central_point):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    results = model(img_path, points=central_point, labels=[1], device="cuda", verbose=False)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    start = time.time()
    predictor.set_image(img_path)
    predictor.model.set_classes(text=[text_prompt])
    predictor.prompts["text"] = [text_prompt]
    results = predictor()
    latency = (time.time() - start) * 1000
    vram = torch.cuda.memory_allocated() / 1024**2 - vram_before if torch.cuda.is_available() else 0
    return results, latency, vram

In [4]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

SAM 1 BASE

In [5]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

SAM 1 GRANDE

In [6]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

## **SAM 2**

BASE

In [7]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

GRANDE

In [8]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

## **SAM 2.1**

2.1 BASE

In [9]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_b.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2.1_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

2.1 GRANDE

In [10]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_l.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2.1_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

## **SAM 3**

In [11]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores   = results[0].boxes.conf.cpu().numpy()
            best_idx = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam3"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must

SAM 3 USANDO PROMPTS PARA SEGMENTAR

In [12]:
import numpy as np
from ultralytics.models.sam import SAM3SemanticPredictor
import cv2
import pandas as pd
import os
import torch
import time

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

TEXT_PROMPT = "skin lesion"

def evaluate_model():
    overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model="C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt",
        device="cuda",
    )
    predictor = SAM3SemanticPredictor(overrides=overrides)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem  = mask_filename.replace("_Segmentation.png", "")
            img_path  = os.path.join(images_dir, img_stem + ".jpg")
            mask_path = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)
            if gt_mask.sum() == 0:
                continue

            results, latency, vram = measure_inference_sam3_prompt(predictor, img_path, TEXT_PROMPT)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            masks = results[0].masks.data.cpu().numpy()
            H, W  = gt_mask.shape

            if len(masks) > 1:
                ious = []
                for m in masks:
                    m_resized    = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                    intersection = np.logical_and(m_resized, gt_mask).sum()
                    union        = np.logical_or(m_resized, gt_mask).sum()
                    ious.append(intersection / union if union > 0 else 0)
                best_idx = np.argmax(ious)
            else:
                best_idx = 0

            pred_mask = masks[best_idx]
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            pred_mask, gt_mask = resize_for_hausdorff(pred_mask, gt_mask)
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":              ["sam3_prompt"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode="a", header=False, index=False)
    else:
        df.to_csv(output_path, index=False)

evaluate_model()

Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\ISIC_2016\ISBI2016_ISIC_Part1_Training_Data\ISIC_0000000.jpg: 644x644 3 skin lesions, 375.8ms
Speed: 3.1ms preprocess, 375.8ms inference, 60.9ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict26
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\ISIC_2016\ISBI2016_ISIC_Part1_Training_Data\ISIC_0000001.jpg: 644x644 1 skin lesion, 1654.8ms
Speed: 2.9ms preprocess, 1654.8ms inference, 4.0ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict26
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fi